# 07 — Cross-Model Evaluation & Error Analysis
**WikiArt Painting Classifier** — NOVA IMS Deep Learning 2025/2026

This notebook evaluates **all trained models** (Baseline CNN, Custom CNN, ViT-B/16) on the held-out test set and produces:
1. Overall accuracy, F1-macro, and per-class metrics for each model
2. Normalised confusion matrices
3. Per-class F1 bar charts
4. Learning curves (training vs validation)
5. Cross-model comparison chart
6. Error analysis — qualitative review of misclassified samples

All figures are saved to `results/figures/` for the report.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
from src.data_loader import build_datasets
from src.evaluate import (
    load_test_data,
    collect_predictions,
    compute_metrics,
    plot_confusion_matrix,
    plot_per_class_f1,
    plot_learning_curves,
    plot_model_comparison,
)

FIGURES_DIR = "../results/figures"
LOGS_DIR = "../results/logs"
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Configuration
Define models to evaluate with their checkpoint paths and training log CSVs.

In [ ]:
MODELS = {
    "baseline": {
        "checkpoint": "../results/models/baseline_cnn.keras",
        "log_csv": f"{LOGS_DIR}/baseline.csv",
    },
    "custom_cnn": {
        "checkpoint": "../results/models/custom_cnn.keras",
        "log_csv": f"{LOGS_DIR}/custom_cnn.csv",
    },
    "transfer": {
        "checkpoint": "../results/models/transfer_model_1.keras",
        "log_csv": f"{LOGS_DIR}/transfer_model_1.csv",
    },
    "vit": {
        "checkpoint": "../results/models/vit.keras",
        "log_csv": f"{LOGS_DIR}/vit.csv",
    },
}

# Verify all files exist
for name, cfg in MODELS.items():
    for key, path in cfg.items():
        status = "OK" if os.path.exists(path) else "MISSING"
        print(f"  [{status}] {name}.{key}: {path}")

## 2. Load Test Data

In [ ]:
test_ds, test_ds_oh, NUM_CLASSES, artist_names = load_test_data(
    splits_dir="../data/splits",
    batch_size=32,
    processed_dir="../data/processed",
)

test_df = pd.read_csv("../data/splits/test.csv", encoding="utf-8-sig")
print(f"Test set: {len(test_df)} images, {NUM_CLASSES} classes")
print(f"Artists: {artist_names[:5]} ... ({len(artist_names)} total)")

## 3. Evaluate Each Model
Loop over all models: load checkpoint, compute predictions, gather metrics, and generate plots.

In [ ]:
all_results = {}  # model_name -> {accuracy, f1_macro, per_class_f1, y_true, y_pred}

# Custom objects needed to load saved models:
# - transfer model: Lambda wrapping resnet50.preprocess_input
# - vit model: CosineWarmup LR schedule in AdamW optimizer
from src.models.vit import CosineWarmup

custom_objects = {
    "preprocess_input": tf.keras.applications.resnet50.preprocess_input,
    "CosineWarmup": CosineWarmup,
}

for model_name, cfg in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  Evaluating: {model_name}")
    print(f"{'='*60}")

    if not os.path.exists(cfg["checkpoint"]):
        print(f"  SKIPPED — checkpoint not found: {cfg['checkpoint']}")
        continue

    # Load model
    with tf.keras.utils.custom_object_scope(custom_objects):
        model = tf.keras.models.load_model(cfg["checkpoint"])
    print(f"  Parameters: {model.count_params():,}")

    # Predictions
    y_true, y_pred = collect_predictions(model, test_ds)

    # Metrics
    metrics = compute_metrics(y_true, y_pred, artist_names)
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1-Macro:  {metrics['f1_macro']:.4f}\n")
    print(metrics["report"])

    all_results[model_name] = {
        **metrics,
        "y_true": y_true,
        "y_pred": y_pred,
    }

    # Free memory
    del model
    tf.keras.backend.clear_session()

## 4. Confusion Matrices

In [ ]:
for model_name, res in all_results.items():
    plot_confusion_matrix(
        res["y_true"], res["y_pred"],
        artist_names, model_name,
        save_dir=FIGURES_DIR,
    )
    plt.show()

## 5. Per-class F1 Scores

In [ ]:
for model_name, res in all_results.items():
    plot_per_class_f1(
        res["per_class_f1"], artist_names,
        model_name, save_dir=FIGURES_DIR,
    )
    plt.show()

## 6. Learning Curves

In [ ]:
for model_name, cfg in MODELS.items():
    if os.path.exists(cfg["log_csv"]):
        plot_learning_curves(cfg["log_csv"], model_name, save_dir=FIGURES_DIR)
        plt.show()
    else:
        print(f"  No log CSV for {model_name}, skipping learning curves.")

## 7. Cross-Model Comparison

In [ ]:
summary_rows = []
for model_name, res in all_results.items():
    summary_rows.append({
        "model": model_name,
        "accuracy": res["accuracy"],
        "f1_macro": res["f1_macro"],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

plot_model_comparison(summary_df, save_dir=FIGURES_DIR)
plt.show()

## 8. Error Analysis
Qualitative review of misclassified samples for the **best model** — which artists are most often confused, and sample images where the model fails.

In [ ]:
# Identify the best model by F1-macro
best_model = summary_df.loc[summary_df["f1_macro"].idxmax(), "model"]
print(f"Best model: {best_model} (F1-macro = {all_results[best_model]['f1_macro']:.4f})")

y_true = all_results[best_model]["y_true"]
y_pred = all_results[best_model]["y_pred"]

# Most confused pairs
from sklearn.metrics import confusion_matrix as sk_cm

cm = sk_cm(y_true, y_pred)
np.fill_diagonal(cm, 0)  # zero out correct predictions

# Top 10 most confused (true_class, predicted_class) pairs
n_top = 10
flat_idx = np.argsort(cm.ravel())[::-1][:n_top]
rows, cols = np.unravel_index(flat_idx, cm.shape)

print(f"\nTop {n_top} most confused artist pairs ({best_model}):")
print(f"{'True Artist':<28} {'Predicted As':<28} {'Count':>5}")
print("-" * 65)
for r, c in zip(rows, cols):
    if cm[r, c] > 0:
        print(f"{artist_names[r]:<28} {artist_names[c]:<28} {cm[r, c]:>5}")

In [ ]:
# Show sample misclassified images from the test set
misclassified_idx = np.where(y_true != y_pred)[0]
print(f"Total misclassified: {len(misclassified_idx)} / {len(y_true)} "
      f"({len(misclassified_idx)/len(y_true)*100:.1f}%)\n")

# Map test_df rows to misclassified samples
test_df_reset = test_df.reset_index(drop=True)
n_samples = min(12, len(misclassified_idx))
sample_idx = np.random.default_rng(42).choice(misclassified_idx, size=n_samples, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, idx in zip(axes.flat, sample_idx):
    row = test_df_reset.iloc[idx]
    img_path = os.path.join("..", row["filepath"])

    # Try processed directory first, fall back to raw
    processed_path = img_path.replace("data/raw", "data/processed").replace("data\\raw", "data\\processed")
    load_path = processed_path if os.path.exists(processed_path) else img_path

    try:
        img = plt.imread(load_path)
        ax.imshow(img)
    except Exception:
        ax.text(0.5, 0.5, "Image\nnot found", ha="center", va="center", transform=ax.transAxes)

    true_name = artist_names[y_true[idx]]
    pred_name = artist_names[y_pred[idx]]
    ax.set_title(f"True: {true_name}\nPred: {pred_name}", fontsize=8, color="red")
    ax.axis("off")

plt.suptitle(f"Sample Misclassified Images — {best_model}", fontsize=14)
plt.tight_layout()

save_path = os.path.join(FIGURES_DIR, f"{best_model}_error_analysis.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

## 9. Summary
Final summary table for the report.

In [ ]:
print("\n" + "=" * 50)
print("  FINAL RESULTS SUMMARY (Test Set)")
print("=" * 50)
print(f"{'Model':<15} {'Accuracy':>10} {'F1-Macro':>10}")
print("-" * 37)
for _, row in summary_df.iterrows():
    print(f"{row['model']:<15} {row['accuracy']:>10.4f} {row['f1_macro']:>10.4f}")
print("-" * 37)
best = summary_df.loc[summary_df["f1_macro"].idxmax()]
print(f"\nBest model: {best['model']} (F1-macro = {best['f1_macro']:.4f})")
print(f"\nAll figures saved to: {os.path.abspath(FIGURES_DIR)}")